# Descarte Fácil — previsão de demanda de remoção de grandes objetos

**Pergunta:** é possível prever a quantidade de solicitações de remoção de grandes objetos por distrito e mês, usando o histórico público do SP156?

**Fonte:** Portal de Dados Abertos da Prefeitura de São Paulo — Dados do SP156.
O notebook reúne arquivos CSV trimestrais oficiais para ampliar o histórico, em vez de utilizar somente o 3º trimestre de 2026.

Página oficial e dicionário: https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156

> O SP156 registra solicitações feitas pelos cidadãos, não todos os descartes irregulares existentes. O modelo prevê demanda registrada, não a quantidade real de resíduos nas ruas.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost requests

In [ ]:
import io, requests, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

# Arquivos trimestrais oficiais identificados na página do conjunto Dados do SP156.
# A Prefeitura publica uma extração por trimestre. Os links podem ser atualizados
# quando o portal republicar os arquivos; consulte a página oficial se houver erro.
BASE = "https://dados.prefeitura.sp.gov.br/dataset/0aecfa2b-aa3a-40d4-8183-0d4351b7fd0a/resource/{}/download/{}"
SOURCES = [
    ("2024-T4", "ed357147-f15c-4bf0-ad15-edc963a9f130", "arquivo-final-4-trim.-2024.csv"),
    ("2025-T1", "7c5aeb58-2cad-40ba-9e9e-af1c17cc5070", "arquivo-final-1-trim-25.csv"),
    ("2025-T2", "f5df7f4a-d614-4f85-b4a0-263ec9370cd9", "arquivo-final-2-trim-25.csv"),
    ("2025-T3", "3b1d2648-d586-40c8-819c-24fbb8005343", "arquivo-final-3-trim-25.csv"),
    ("2025-T4", "d3a98097-8128-45d4-b8fe-21874b81c971", "arquivo-final-4-trim-25.csv"),
    ("2026-T1", "75105148-4efd-48f3-8ae0-fc6c2adf7060", "arquivo-final-1-trim-26.csv"),
    ("2026-T2", "368eb32d-ef25-4441-be8f-fce0146575a1", "arquivo-final-2-trim-26.csv"),
    ("2026-T3", "f67fcc12-8da4-485f-92a3-d2af40cc771e", "arquivo-final-3-trim-26.csv"),
]

frames, download_log = [], []
for quarter, resource_id, filename in SOURCES:
    url = BASE.format(resource_id, filename)
    response = requests.get(url, timeout=180)
    if response.status_code != 200:
        print(f"Aviso: {quarter} retornou HTTP {response.status_code}; confira o link no portal.")
        continue
    frame = pd.read_csv(io.BytesIO(response.content), sep=None, engine="python", encoding_errors="replace")
    frame["arquivo_origem"] = quarter
    frames.append(frame)
    download_log.append({"trimestre": quarter, "linhas": len(frame), "colunas": len(frame.columns)})
    print(f"{quarter}: {len(frame):,} registros")

if not frames:
    raise RuntimeError("Nenhum CSV foi carregado. Confira os links e a conexão com o Portal de Dados Abertos.")

expected_quarters = [item[0] for item in SOURCES]
loaded_quarters = [item["trimestre"] for item in download_log]
missing_quarters = sorted(set(expected_quarters) - set(loaded_quarters))
if missing_quarters:
    raise RuntimeError(
        "Não foi possível carregar todos os trimestres esperados: "
        + ", ".join(missing_quarters)
        + ". A execução foi interrompida para não preencher períodos ausentes artificialmente com zeros."
    )

raw = pd.concat(frames, ignore_index=True, sort=False)
print("\nBase combinada:", raw.shape)
display(pd.DataFrame(download_log))
display(raw.head())
print(raw.columns.tolist())

### Observação metodológica

O portal informa que as solicitações publicadas cobrem o período de 2012 a 30/09/2026 e são disponibilizadas em arquivos trimestrais. Este notebook combina os arquivos trimestrais disponíveis listados na célula de coleta. A Prefeitura alerta que serviços podem mudar de nome, ser reclassificados e receber protocolos incluídos em republicações.

A coluna Data de Abertura aparece em formato mês/dia/ano (MM/DD/YYYY) nos arquivos recentes. O notebook verifica a taxa de conversão e interrompe a execução se a leitura das datas for insuficiente. A unidade territorial utilizada é **Distrito**, não Subprefeitura.

## 1. Preparação e seleção do serviço

A análise utiliza as colunas de data, serviço e distrito. Como os arquivos são publicados em períodos diferentes, primeiro verificamos se essas colunas existem na base combinada.

In [ ]:
date_col = "Data de Abertura"
service_col = "Serviço"
district_col = "Distrito"

missing = [c for c in [date_col, service_col, district_col] if c not in raw.columns]
if missing:
    raise ValueError(f"Colunas esperadas não encontradas: {missing}. Confira o dicionário e raw.columns.")

print("Data:", date_col, "| Serviço:", service_col, "| Unidade territorial:", district_col)
print("Arquivos incluídos:", raw["arquivo_origem"].nunique())
print(raw["arquivo_origem"].value_counts().sort_index().to_string())

In [ ]:
df = raw.copy()
date_values = df[date_col].astype(str).str.strip()

# O formato MM/DD/YYYY foi confirmado nos arquivos recentes. Tentamos esse
# padrão primeiro; se a cobertura for baixa, testamos o padrão brasileiro.
parsed = pd.to_datetime(date_values, format="%m/%d/%Y", errors="coerce")
if parsed.notna().mean() < 0.95:
    alternative = pd.to_datetime(date_values, format="%d/%m/%Y", errors="coerce")
    if alternative.notna().mean() > parsed.notna().mean():
        parsed = alternative

df[date_col] = parsed
valid_dates = df[date_col].notna().mean()
print(f"Proporção de datas interpretadas: {valid_dates:.1%}")
if valid_dates < 0.95:
    raise ValueError("Menos de 95% das datas foram interpretadas. Verifique os formatos dos CSVs.")

df = df.dropna(subset=[date_col, district_col, service_col]).copy()
print("Período:", df[date_col].min(), "a", df[date_col].max())
print("\nServiços com termos de interesse:")
service_counts = df.loc[
    df[service_col].astype(str).str.contains("entulho|grandes objetos|remoção|remocao", case=False, na=False),
    service_col
].value_counts()
print(service_counts.head(60).to_string())

## 2. Agregação temporal e variável-alvo

Selecionamos os serviços relacionados a entulho e grandes objetos. A unidade de análise será **distrito-mês**; o alvo é a quantidade de solicitações registradas naquele mês. A seleção deve ser conferida na lista de serviços impressa acima, pois a nomenclatura pode variar entre períodos.

In [ ]:
TERMS = ["grandes objetos", "entulho e grandes objetos"]
service_text = df[service_col].astype(str).str.lower()
selected = df[service_text.apply(lambda x: any(term in x for term in TERMS))].copy()

if selected.empty:
    raise ValueError("Nenhum registro encontrado. Revise os nomes de serviço exibidos na célula anterior.")

selected["mes"] = selected[date_col].dt.to_period("M").dt.to_timestamp()
agg = (
    selected.groupby([district_col, "mes"]).size()
    .rename("solicitacoes").reset_index()
    .rename(columns={district_col: "regiao"})
)

regions = sorted(agg["regiao"].dropna().unique())
months = pd.date_range(agg["mes"].min(), agg["mes"].max(), freq="MS")
panel = pd.MultiIndex.from_product([regions, months], names=["regiao", "mes"]).to_frame(index=False)
panel = panel.merge(agg, on=["regiao", "mes"], how="left")
panel["solicitacoes"] = panel["solicitacoes"].fillna(0).astype(int)

print("Registros selecionados:", len(selected))
print("Distritos:", len(regions))
print("Período:", panel["mes"].min(), "a", panel["mes"].max())
print("Meses:", panel["mes"].nunique(), "| Linhas do painel:", len(panel))
display(panel.head())

## 3. Engenharia de atributos e divisão temporal

Utilizamos variáveis de calendário e defasagens de 1, 2 e 3 meses. A divisão respeita a ordem temporal: os seis meses mais recentes ficam reservados para teste, os seis anteriores para validação e todo o período anterior para treino. Assim, validação e teste não compartilham meses com o treino.

In [ ]:
panel = panel.sort_values(["regiao", "mes"]).copy()
panel["ano"] = panel["mes"].dt.year
panel["mes_num"] = panel["mes"].dt.month
panel["trimestre"] = panel["mes"].dt.quarter

for lag in [1, 2, 3]:
    panel[f"lag_{lag}"] = panel.groupby("regiao")["solicitacoes"].shift(lag)

feature_cols = ["regiao", "ano", "mes_num", "trimestre", "lag_1", "lag_2", "lag_3"]
model_df = panel.dropna(subset=["lag_1", "lag_2", "lag_3"]).copy()
X = pd.get_dummies(model_df[feature_cols], columns=["regiao"], drop_first=False, dtype=int)
y = model_df["solicitacoes"]

months = np.array(sorted(model_df["mes"].unique()))
if len(months) < 18:
    raise ValueError(
        f"Histórico insuficiente após as defasagens: {len(months)} meses. "
        "São necessários pelo menos 18 meses para treino, validação e teste temporais."
    )

test_months = months[-6:]
val_months = months[-12:-6]
train_months = months[:-12]

train_mask = model_df["mes"].isin(train_months)
val_mask = model_df["mes"].isin(val_months)
test_mask = model_df["mes"].isin(test_months)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_trainval, y_trainval = X.loc[train_mask | val_mask], y.loc[train_mask | val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(f"Treino: {train_months[0]} a {train_months[-1]} | {len(X_train):,} linhas")
print(f"Validação: {val_months[0]} a {val_months[-1]} | {len(X_val):,} linhas")
print(f"Teste: {test_months[0]} a {test_months[-1]} | {len(X_test):,} linhas")
print("Meses distintos — treino/validação/teste:", len(train_months), len(val_months), len(test_months))

## 4. Treinamento e comparação de hiperparâmetros
Comparamos uma linha de base (previsão igual ao mês anterior), Random Forest e XGBoost. Métricas: MAE, RMSE e R². O MAE é especialmente interpretável: erro médio em número de solicitações.

In [ ]:
# A linha de base prevê que o próximo mês repetirá o volume do mês anterior.
results, fitted = [], {}

baseline = model_df.loc[test_mask, "lag_1"].values
results.append({
    "modelo": "Baseline lag-1",
    "MAE": mean_absolute_error(y_test, baseline),
    "RMSE": mean_squared_error(y_test, baseline) ** 0.5,
    "R2": r2_score(y_test, baseline)
})

# Seleção de hiperparâmetros apenas na validação temporal.
rf_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [8, None],
    "min_samples_leaf": [1, 3]
}))
best_rf_params, best_rf_mae = None, float("inf")
for params in rf_grid:
    candidate = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    score = mean_absolute_error(y_val, val_pred)
    if score < best_rf_mae:
        best_rf_params, best_rf_mae = params, score

rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_rf_params)
rf.fit(X_trainval, y_trainval)
fitted["Random Forest"] = rf
rf_pred = rf.predict(X_test)
results.append({
    "modelo": f"Random Forest {best_rf_params}",
    "MAE": mean_absolute_error(y_test, rf_pred),
    "RMSE": mean_squared_error(y_test, rf_pred) ** 0.5,
    "R2": r2_score(y_test, rf_pred)
})

xgb_grid = list(ParameterGrid({
    "n_estimators": [200, 400],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.08]
}))
best_xgb_params, best_xgb_mae = None, float("inf")
for params in xgb_grid:
    candidate = XGBRegressor(
        random_state=42, objective="reg:squarederror", n_jobs=-1, **params
    )
    candidate.fit(X_train, y_train)
    val_pred = candidate.predict(X_val)
    score = mean_absolute_error(y_val, val_pred)
    if score < best_xgb_mae:
        best_xgb_params, best_xgb_mae = params, score

xgb = XGBRegressor(
    random_state=42, objective="reg:squarederror", n_jobs=-1, **best_xgb_params
)
xgb.fit(X_trainval, y_trainval)
fitted["XGBoost"] = xgb
xgb_pred = xgb.predict(X_test)
results.append({
    "modelo": f"XGBoost {best_xgb_params}",
    "MAE": mean_absolute_error(y_test, xgb_pred),
    "RMSE": mean_squared_error(y_test, xgb_pred) ** 0.5,
    "R2": r2_score(y_test, xgb_pred)
})

metrics = pd.DataFrame(results).sort_values("MAE")
display(metrics)

In [ ]:
best_name = metrics.loc[metrics["modelo"].str.startswith(("Random Forest", "XGBoost")), "modelo"].iloc[0]
best_model = fitted["Random Forest"] if best_name.startswith("Random Forest") else fitted["XGBoost"]
best_pred = best_model.predict(X_test)

plt.figure(figsize=(7,5))
plt.scatter(y_test, best_pred, alpha=.35)
lim = max(float(y_test.max()), float(np.max(best_pred)))
plt.plot([0,lim],[0,lim],"--")
plt.xlabel("Solicitações reais")
plt.ylabel("Solicitações previstas")
plt.title("Valores reais x previstos — melhor modelo")
plt.tight_layout()
plt.show()

if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values().tail(12)
    imp.plot(kind="barh", figsize=(8,5), title="Atributos mais importantes")
    plt.tight_layout()
    plt.show()


## 5. Interpretação para o cidadão

Após executar o notebook com os arquivos trimestrais, registre:
- Melhor modelo e hiperparâmetros;
- MAE, RMSE e R² no período de teste;
- Comparação com o baseline;
- Atributos mais relevantes;
- Período efetivamente coberto pelos arquivos que carregaram com sucesso.

**Aplicação no Descarte Fácil:** a previsão pode apoiar ações de comunicação e planejamento em distritos com maior demanda registrada esperada. Deve ser apresentada como indicador agregado, sem substituir o calendário oficial do Cata-Bagulho ou prometer uma data de coleta.

**Limitações:** solicitações SP156 dependem da iniciativa dos cidadãos e não representam todos os descartes. Serviços podem mudar de nome, ser reclassificados e receber registros tardios. O modelo prevê demanda registrada por distrito, não por CEP/endereço, e a avaliação é uma previsão mensal de um passo à frente, utilizando os meses anteriores observados.